# Multi-Agent Systems with Handoffs — OpenAI Agents SDK

This notebook explains and demonstrates the **Handoffs pattern** for building multi-agent systems with the OpenAI Agents SDK.

## Learning objectives
- Understand why multi-agent systems are useful.
- Compare **Handoffs** and **Manager** patterns.
- Build specialist agents for different domains.
- Build a triage agent that routes questions to specialists.
- Use `handoffs` to transfer control between agents.
- Run the system synchronously and inspect the final answer.


## 1. What is a Multi-Agent System?

A single agent may be enough for simple tasks, but complex applications often benefit from multiple specialized agents.

Instead of creating one large agent that knows how to handle everything, we can create several focused agents. Each agent has a specific responsibility and instructions.

For example:
- A **Math Tutor** handles mathematical questions.
- A **History Tutor** handles historical questions.
- A **Triage Agent** determines which specialist should handle the user's request.

This separation makes agent behavior easier to understand, maintain, and extend.

## 2. Two Main Multi-Agent Patterns

There are two important patterns to understand.

### Pattern A — Handoffs / Decentralized Model

A triage agent receives the user's request and decides which specialist should take over. After the handoff, the specialist becomes responsible for completing the task.

Conceptually:

`User → Triage Agent → Specialist Agent → Final Answer`

The specialist can see the conversation context and takes ownership of the interaction.

A good use case is customer support:
- Billing Agent
- Shipping Agent
- Technical Support Agent

### Pattern B — Manager / Centralized Model

A central manager remains in control. It delegates work to specialist agents, often by using agents as tools, and then combines their results into a final response.

Conceptually:

`User → Manager → Specialist 1 / Specialist 2 / Specialist 3 → Manager → Final Answer`

A good use case is a project manager agent coordinating a researcher, writer, and reviewer.

### Rule of thumb

- Use **Handoffs** when you want to route a conversation to the appropriate specialist.
- Use the **Manager pattern** when one central agent needs to coordinate multiple specialists and remain in control.

## 3. Install the OpenAI Agents SDK

Install the package if it is not already available in your environment.

In [ ]:
!pip install -U openai-agents

## 4. Configure Your OpenAI API Key

The Agents SDK reads the API key from the `OPENAI_API_KEY` environment variable.

For security, do not hard-code your API key directly into a notebook that you plan to share.

In [ ]:
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Please set the OPENAI_API_KEY environment variable before running the agent.")
else:
    print("OPENAI_API_KEY is available.")

## 5. Import the Agents SDK

We use:
- `Agent` to define an agent.
- `Runner` to execute the agent loop.

In [ ]:
from agents import Agent, Runner

## 6. Define the Math Specialist Agent

The first specialist is a Math Tutor. Its job is to answer mathematical questions clearly and explain the reasoning step by step.

The `handoff_description` is important. It helps the triage agent understand when this specialist should receive a handoff.

In [ ]:
math_agent = Agent(
    name="Math Tutor",
    model="gpt-5.5",
    instructions=(
        "You are a helpful math tutor. "
        "Solve mathematical questions accurately and explain the reasoning clearly. "
        "Use simple language and show the important calculation steps."
    ),
    handoff_description=(
        "Handles mathematics questions, calculations, percentages, "
        "algebra, arithmetic, and other quantitative problems."
    ),
)

## 7. Define the History Specialist Agent

The second specialist is a History Tutor. It handles questions about historical events, people, civilizations, and historical context.

In [ ]:
history_agent = Agent(
    name="History Tutor",
    model="gpt-5.5",
    instructions=(
        "You are a helpful history tutor. "
        "Answer historical questions accurately and provide useful context. "
        "Explain important dates, people, events, and causes when relevant."
    ),
    handoff_description=(
        "Handles history questions about historical events, civilizations, "
        "wars, important people, places, and historical context."
    ),
)

## 8. Define the Triage Agent

The triage agent is responsible for routing the user's question to the correct specialist.

The key property is `handoffs`.

Each specialist listed in `handoffs` becomes available as a handoff destination. The triage agent can transfer control to the appropriate specialist.

In [ ]:
triage_agent = Agent(
    name="Triage Agent",
    model="gpt-5.5",
    instructions=(
        "You are a triage agent. "
        "Determine which specialist is best suited to answer the user's question. "
        "Hand off mathematics questions to the Math Tutor. "
        "Hand off history questions to the History Tutor. "
        "Do not attempt to answer specialist questions yourself when a specialist is available."
    ),
    handoffs=[math_agent, history_agent],
)

## 9. Run a Math Question Through the Handoff System

The user asks a mathematics question.

Expected flow:

`User → Triage Agent → Math Tutor → Final Answer`

The triage agent should recognize that the question is mathematical and hand off control to the Math Tutor.

In [ ]:
math_question = "What is 15% of 240?"

result = Runner.run_sync(triage_agent, math_question)

print("Question:", math_question)
print("Answer:", result.final_output)

## 10. Run a History Question Through the Handoff System

Now we ask a historical question.

Expected flow:

`User → Triage Agent → History Tutor → Final Answer`

The triage agent should recognize that this is a history question and hand off control to the History Tutor.

In [ ]:
history_question = "Tell me about the Great Wall of China and why it was built."

result = Runner.run_sync(triage_agent, history_question)

print("Question:", history_question)
print("Answer:", result.final_output)

## 11. Test Multiple Questions

The same triage agent can route different questions to different specialists.

In [ ]:
questions = [
    "What is 25% of 800?",
    "Who built the Great Wall of China?",
    "If I multiply 15 by 8 and divide the result by 3, what do I get?",
    "What were the major causes of World War I?",
]

for question in questions:
    result = Runner.run_sync(triage_agent, question)
    print("=" * 80)
    print("Question:", question)
    print("Answer:", result.final_output)

## 12. Understand the Handoff Flow

The important architecture is:

1. The user sends a question to the **Triage Agent**.
2. The Triage Agent reasons about the user's request.
3. It determines which specialist is appropriate.
4. The SDK performs the handoff to that specialist.
5. The specialist handles the request.
6. The specialist produces the final response.

In this example:

- Math question → **Math Tutor**
- History question → **History Tutor**

Under the hood, the handoff destinations are exposed to the triage agent as mechanisms it can use to transfer control.

## 13. Handoffs vs. Manager Pattern

| Feature | Handoffs | Manager Pattern |
|---|---|---|
| Control | Specialist takes over | Manager remains in control |
| Architecture | Decentralized | Centralized |
| Best for | Routing conversations | Coordinating multiple specialists |
| User interaction | Specialist can continue the conversation | User primarily interacts with manager |
| Example | Customer support routing | Project manager coordinating research and writing |

### Handoffs

Use handoffs when the user's request should be transferred to one specialist that can own the rest of the interaction.

### Manager

Use a manager when a central agent needs to delegate work to several specialists, collect their results, and synthesize the final answer.

## 14. Key Takeaways

### 1. Multi-agent systems divide responsibilities
Instead of one agent doing everything, specialized agents can focus on specific domains.

### 2. Handoffs are useful for routing
A triage agent can determine which specialist should take over a conversation.

### 3. `handoff_description` helps routing
The description explains when a specialist should receive a handoff.

### 4. `handoffs` defines available destinations
The triage agent receives a list of specialist agents that it can hand off to.

### 5. The Manager pattern is different
The manager stays in control and uses specialists as tools rather than transferring ownership of the conversation.

### Final mental model

**Handoffs:**
`Triage → Specialist takes over`

**Manager:**
`Manager → Calls specialists → Manager synthesizes result`
